In [1]:
import os
import sys
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))
import scripts.utils as utils
%reload_ext autoreload
%autoreload 2

df_raw_occupation = utils.extract("by_occupation_prior_to_migration.xls")
df_raw_occupation


,MAJOR OCCUPATION GROUP,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2013,2014,2015,2016,2017,2018,2019,2020,TOTAL,%
0,A. EMPLOYED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Prof'l, Tech'l, & Related Workers",4821.0,5509.0,3928.0,3791.0,3869.0,4147.0,4899.0,7689,6861,...,6499.0,6507.0,7504.0,6781.0,5074.0,5029.0,5209.0,1240.0,254748.0,0.101262
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Managerial, Executive, and",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Administrative Workers,451.0,360.0,324.0,320.0,366.0,369.0,420.0,678,480,...,2195.0,1905.0,1789.0,1554.0,1258.0,1211.0,1249.0,282.0,40349.0,0.016039
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Clerical Workers,2475.0,3992.0,2796.0,3131.0,2972.0,3394.0,4605.0,2157,1681,...,1915.0,1954.0,1925.0,1878.0,1453.0,1370.0,1385.0,356.0,81404.0,0.032358
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Sales Workers,1628.0,1865.0,1414.0,1326.0,1819.0,2109.0,2825.0,2184,2251,...,2129.0,2495.0,2807.0,3018.0,2287.0,2383.0,2366.0,498.0,99538.0,0.039566


In [2]:
df_clean_occupation = utils.clean(df_raw_occupation)

row_mapping = {
    "Managerial, Executive, and": "Managerial, Executive, and Administrative Workers",
    "Administrative Workers": "Managerial, Executive, and Administrative Workers", 
    "Agri, Animal Husbandry, Forestry": "Agri, Animal Husbandry, Forestry, Workers & Fishermen", 
    "Workers & Fishermen": "Agri, Animal Husbandry, Forestry, Workers & Fishermen",
    "Production Process, Transport": "Production Process, Transport, Equipment Operators, & Laborers",
    "Equipment Operators, & Laborers": "Production Process, Transport, Equipment Operators, & Laborers"
}
target_groups = ["A. EMPLOYED", "B. UNEMPLOYED"]

df_clean_occupation["Employment Group"] = np.where(
    df_clean_occupation["MAJOR OCCUPATION GROUP"].isin(target_groups),
    df_clean_occupation["MAJOR OCCUPATION GROUP"],
    np.nan
)
df_clean_occupation["Employment Group"] = df_clean_occupation["Employment Group"].ffill()
df_clean_occupation = df_clean_occupation[df_clean_occupation["MAJOR OCCUPATION GROUP"] != df_clean_occupation["Employment Group"]].copy()
df_clean_occupation.loc[df_clean_occupation["MAJOR OCCUPATION GROUP"] == "No Occupation Reported", "Employment Group"] = "C. NOT REPORTED"
df_clean_occupation["Employment Group"] = df_clean_occupation["Employment Group"].str.split(".").str[1]

df_clean_occupation["MAJOR OCCUPATION GROUP"] = df_clean_occupation["MAJOR OCCUPATION GROUP"].replace(row_mapping)
df_clean_occupation = df_clean_occupation.groupby("MAJOR OCCUPATION GROUP", as_index=False).sum()

df_clean_occupation["Employment Group"] = df_clean_occupation["Employment Group"].str.strip().replace({"EMPLOYED EMPLOYED": "EMPLOYED"})

df_clean_occupation


,MAJOR OCCUPATION GROUP,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2014,2015,2016,2017,2018,2019,2020,TOTAL,%,Employment Group
0,"Agri, Animal Husbandry, Forestry, Workers & Fi...",2238,1894,1787,1475,1483,1389,1105,1227,1081,...,858,1151,1032,920,789,590,136,44888.0,0.017843,EMPLOYED
1,Clerical Workers,2475,3992,2796,3131,2972,3394,4605,2157,1681,...,1954,1925,1878,1453,1370,1385,356,81404.0,0.032358,EMPLOYED
2,Housewives,11202,12469,9025,8730,9247,8695,9315,9720,9929,...,13423,13723,13357,11400,11310,9326,2918,485115.0,0.192833,UNEMPLOYED
3,"Managerial, Executive, and Administrative Workers",451,360,324,320,366,369,420,678,480,...,1905,1789,1554,1258,1211,1249,282,40349.0,0.016039,EMPLOYED
4,Members of the Armed Forces,858,504,96,57,36,32,73,315,309,...,149,285,214,190,220,259,54,8078.0,0.003211,EMPLOYED
5,Minors (Below 7 years old),3625,4419,3826,3919,4348,4750,5382,5683,5114,...,5849,6159,5788,4555,4258,3687,848,190528.0,0.075735,UNEMPLOYED
6,No Occupation Reported,5204,5540,4620,5841,7566,8755,9998,8769,9407,...,18499,21592,21374,20001,19096,16921,3618,419523.0,0.166760,NOT REPORTED
7,Out of School Youth,0,0,0,0,0,0,0,132,222,...,435,473,1466,564,443,515,126,11502.0,0.004572,UNEMPLOYED
8,"Production Process, Transport, Equipment Opera...",1780,2758,1951,1992,1848,1822,1754,2281,2236,...,2290,2708,2494,2342,1819,1597,405,87491.0,0.034778,EMPLOYED
9,"Prof'l, Tech'l, & Related Workers",4821,5509,3928,3791,3869,4147,4899,7689,6861,...,6507,7504,6781,5074,5029,5209,1240,254748.0,0.101262,EMPLOYED


In [3]:
# build dimension table
unique_occupations = (
    df_clean_occupation[["MAJOR OCCUPATION GROUP", "Employment Group"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_occupations["Occupation_ID"] = [f"PH_OCC_{i+1:03d}" for i in unique_occupations.index]
dim_ph_occupation = (
    unique_occupations[["Occupation_ID", "MAJOR OCCUPATION GROUP", "Employment Group"]]
    .rename(columns={"MAJOR OCCUPATION GROUP": "Occupation_Group", "Employment GROUP": "Employment_Status"})
)

dim_ph_occupation

,Occupation_ID,Occupation_Group,Employment Group
0,PH_OCC_001,"Agri, Animal Husbandry, Forestry, Workers & Fi...",EMPLOYED
1,PH_OCC_002,Clerical Workers,EMPLOYED
2,PH_OCC_003,Housewives,UNEMPLOYED
3,PH_OCC_004,"Managerial, Executive, and Administrative Workers",EMPLOYED
4,PH_OCC_005,Members of the Armed Forces,EMPLOYED
5,PH_OCC_006,Minors (Below 7 years old),UNEMPLOYED
6,PH_OCC_007,No Occupation Reported,NOT REPORTED
7,PH_OCC_008,Out of School Youth,UNEMPLOYED
8,PH_OCC_009,"Production Process, Transport, Equipment Opera...",EMPLOYED
9,PH_OCC_010,"Prof'l, Tech'l, & Related Workers",EMPLOYED


In [4]:
# make fact table

df_fact_prep = df_clean_occupation.merge(
    dim_ph_occupation, left_on="MAJOR OCCUPATION GROUP", right_on="Occupation_Group", how="left"
)

occupation_cols = [col for col in df_clean_occupation.columns if col in map(str, range(1981, 2021))]
fact_ph_occupation = pd.melt(
    df_fact_prep,
    id_vars=["Occupation_ID"],
    value_vars=occupation_cols,
    var_name="Year",
    value_name="Emigrant_Count"
)

fact_ph_occupation

,Occupation_ID,Year,Emigrant_Count
0,PH_OCC_001,1981,2238
1,PH_OCC_002,1981,2475
2,PH_OCC_003,1981,11202
3,PH_OCC_004,1981,451
4,PH_OCC_005,1981,858
...,...,...,...
595,PH_OCC_011,2020,0
596,PH_OCC_012,2020,592
597,PH_OCC_013,2020,498
598,PH_OCC_014,2020,664


In [5]:
# download file
utils.load(dim_ph_occupation, "dim_ph_occupation.csv")
utils.load(fact_ph_occupation, "fact_ph_occupation.csv")

downloaded csv file dim_ph_occupation.csv
downloaded csv file fact_ph_occupation.csv
